# Escenario 1 — Agente de Resolución de Soporte al Cliente

**Dominios:** Agentic Architecture & Orchestration · Tool Design & MCP Integration · Context Management & Reliability

## Contexto
Construís un agente de soporte que maneja devoluciones, disputas de facturación y problemas de cuenta. El agente tiene acceso a cuatro tools MCP: `get_customer`, `lookup_order`, `process_refund`, `escalate_to_human`. Objetivo: 80%+ de resolución en el primer contacto.

## Lo que vas a implementar
1. Tools con descripciones completas y respuestas de error estructuradas
2. Gate programático: `lookup_order` y `process_refund` bloqueadas hasta que `get_customer` haya corrido
3. Hook `PostToolUse` para normalizar formatos heterogéneos
4. Hook de intercepción para bloquear reembolsos > $500
5. Criterios explícitos de escalación con few-shot examples
6. Agentic loop completo con manejo de stop_reason

In [ ]:
import anthropic
import json
from datetime import datetime

client = anthropic.Anthropic()

## Paso 1 — Definir las tools con descripciones completas

**Concepto clave:** Las descripciones son el mecanismo principal de selección de tools. Deben incluir: propósito, formatos de entrada, cuándo usarla vs. alternativas, qué devuelve, casos edge.

In [ ]:
TOOLS = [
    {
        "name": "get_customer",
        "description": (
            "Retrieves customer account information. ALWAYS call this FIRST before any order "
            "operation to verify customer identity. "
            "Input formats: customer ID (CUST-XXXXX) or email address. "
            "Do NOT use for order details — use lookup_order instead. "
            "Returns: customer_id, name, email, account_status, tier, created_at (Unix timestamp)."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "identifier": {
                    "type": "string",
                    "description": "Customer ID (CUST-XXXXX) or email address"
                }
            },
            "required": ["identifier"]
        }
    },
    {
        "name": "lookup_order",
        "description": (
            "Retrieves order details by order number or tracking number. "
            "Use when the customer asks about a specific order, delivery, or shipment. "
            "Requires a verified customer_id from get_customer first. "
            "Do NOT use for customer account information. "
            "Returns: order_id, status_code (1=pending,2=shipped,3=delivered,4=returned), "
            "total_amount, items, shipping_date."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "order_identifier": {"type": "string"},
                "customer_id": {"type": "string", "description": "Verified customer ID from get_customer"}
            },
            "required": ["order_identifier", "customer_id"]
        }
    },
    {
        "name": "process_refund",
        "description": (
            "Processes a refund for a specific order. "
            "Requires verified customer_id and confirmed order_id. "
            "Maximum refund amount is $500 — escalate to human for higher amounts. "
            "Returns: refund_id, status, estimated_credit_days."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string"},
                "customer_id": {"type": "string"},
                "amount": {"type": "number"},
                "reason": {"type": "string"}
            },
            "required": ["order_id", "customer_id", "amount", "reason"]
        }
    },
    {
        "name": "escalate_to_human",
        "description": (
            "Escalates the case to a human agent. Use when: "
            "(1) customer explicitly requests a human, "
            "(2) refund exceeds $500 limit, "
            "(3) policy doesn't cover the specific situation, "
            "(4) agent cannot make meaningful progress. "
            "Include a structured summary — the human won't have access to the transcript."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": ["string", "null"]},
                "issue_summary": {"type": "string"},
                "root_cause": {"type": "string"},
                "actions_taken": {"type": "array", "items": {"type": "string"}},
                "recommended_action": {"type": "string"},
                "escalation_reason": {
                    "type": "string",
                    "enum": ["customer_requested", "exceeds_limit", "policy_gap", "no_progress"]
                }
            },
            "required": ["issue_summary", "escalation_reason"]
        }
    }
]

## Paso 2 — Implementar los backends de tools (simulados)

**Concepto clave:** Incluir `isError`, `errorCategory`, `isRetryable` en las respuestas de error.

In [ ]:
# Base de datos simulada
CUSTOMERS_DB = {
    "maria@example.com": {
        "customer_id": "CUST-45821",
        "name": "María García",
        "email": "maria@example.com",
        "account_status": "active",
        "tier": "gold",
        "created_at": 1640000000  # Unix timestamp
    }
}

ORDERS_DB = {
    "ORD-9872": {
        "order_id": "ORD-9872",
        "customer_id": "CUST-45821",
        "status_code": 3,  # delivered
        "total_amount": 89.99,
        "items": [{"name": "Auriculares Bluetooth", "qty": 1, "price": 89.99}],
        "shipping_date": "2024-01-10"
    }
}


def tool_get_customer(identifier: str) -> dict:
    customer = CUSTOMERS_DB.get(identifier)
    if not customer:
        return {
            "isError": True,
            "errorCategory": "validation",
            "isRetryable": False,
            "message": f"No customer found with identifier: {identifier}"
        }
    return {"verified": True, **customer}


def tool_lookup_order(order_identifier: str, customer_id: str) -> dict:
    order = ORDERS_DB.get(order_identifier)
    if not order:
        return {"success": True, "results": [], "message": "No order found with that identifier"}
    if order["customer_id"] != customer_id:
        return {
            "isError": True,
            "errorCategory": "permission",
            "isRetryable": False,
            "message": "Order does not belong to this customer"
        }
    return order


def tool_process_refund(order_id: str, customer_id: str, amount: float, reason: str) -> dict:
    # Este check se hace también en el hook, pero puede ejecutarse si el hook no está activo
    if amount > 500:
        return {
            "isError": True,
            "errorCategory": "permission",
            "isRetryable": False,
            "message": f"Refund of ${amount} exceeds agent limit of $500. Escalate to human."
        }
    return {
        "success": True,
        "refund_id": f"REF-{order_id}-001",
        "status": "approved",
        "estimated_credit_days": 3
    }


def tool_escalate_to_human(issue_summary: str, escalation_reason: str, **kwargs) -> dict:
    print(f"\n🔔 ESCALACIÓN A HUMANO")
    print(f"Razón: {escalation_reason}")
    print(f"Resumen: {issue_summary}")
    return {"escalated": True, "ticket_id": "TICKET-789", "estimated_wait_minutes": 5}

## Paso 3 — Gate programático y hooks

**Concepto clave:** El enforcement programático es determinístico. Las instrucciones en el prompt son probabilísticas (~99%). Para reglas de negocio críticas (verificación de identidad, límites financieros) → siempre enforcement programático.

In [ ]:
class SupportAgentState:
    def __init__(self):
        self.customer_verified = False
        self.customer_id = None
    
    def post_tool_use_hook(self, tool_name: str, result: dict) -> dict:
        """Normaliza formatos heterogéneos antes de que el modelo los procese."""
        if tool_name == "get_customer" and result.get("verified"):
            # Normalizar Unix timestamp → ISO 8601
            if "created_at" in result:
                result["created_at"] = datetime.fromtimestamp(result["created_at"]).strftime("%Y-%m-%d")
            # Actualizar estado interno
            self.customer_verified = True
            self.customer_id = result.get("customer_id")
        
        if tool_name == "lookup_order" and result.get("status_code"):
            # Normalizar código numérico → descripción legible
            status_map = {1: "pending", 2: "shipped", 3: "delivered", 4: "returned"}
            result["status"] = status_map.get(result["status_code"], "unknown")
        
        return result
    
    def pre_tool_call_hook(self, tool_name: str, tool_input: dict) -> dict | None:
        """Intercepta llamadas. Retorna None para permitir, dict de error para bloquear."""
        # GATE: bloquear herramientas de orden hasta que el cliente esté verificado
        if tool_name in ["lookup_order", "process_refund"] and not self.customer_verified:
            return {
                "isError": True,
                "errorCategory": "validation",
                "message": "Customer must be verified with get_customer before order operations"
            }
        
        # HOOK: bloquear reembolsos > $500
        if tool_name == "process_refund":
            amount = tool_input.get("amount", 0)
            if amount > 500:
                return {
                    "isError": True,
                    "errorCategory": "permission",
                    "isRetryable": False,
                    "message": f"Refund of ${amount} exceeds $500 agent limit. Use escalate_to_human.",
                    "blocked_by": "REFUND_LIMIT_HOOK"
                }
        
        return None  # Permitir la llamada
    
    def execute_tool(self, tool_name: str, tool_input: dict) -> str:
        # Verificar pre-hook
        block = self.pre_tool_call_hook(tool_name, tool_input)
        if block:
            return json.dumps(block)
        
        # Ejecutar la tool
        tool_fns = {
            "get_customer": tool_get_customer,
            "lookup_order": tool_lookup_order,
            "process_refund": tool_process_refund,
            "escalate_to_human": tool_escalate_to_human
        }
        result = tool_fns[tool_name](**tool_input)
        
        # Aplicar post-hook
        result = self.post_tool_use_hook(tool_name, result)
        
        return json.dumps(result)

## Paso 4 — System prompt con criterios explícitos de escalación

In [ ]:
SYSTEM_PROMPT = """
Sos un agente de soporte al cliente. Tu objetivo es resolver el 80%+ de los casos de forma autónoma.

SECUENCIA OBLIGATORIA:
1. Siempre verificar la identidad del cliente con get_customer PRIMERO.
2. Solo después consultar órdenes con lookup_order.
3. Solo procesar reembolsos con process_refund cuando tenés la orden confirmada.

CUÁNDO ESCALAR (usar escalate_to_human):
- El cliente pide explícitamente hablar con una persona → ESCALAR INMEDIATAMENTE, sin preguntar
- El reembolso supera $500 → ESCALAR con el monto y la razón
- La política no contempla el caso específico (brecha de política) → ESCALAR
- No podés hacer progreso significativo → ESCALAR

CUÁNDO NO ESCALAR:
- El caso es complejo pero está dentro de tus capacidades
- El cliente está frustrado pero el issue es resolvible

Ejemplos de decisión:

Cliente: "Quiero hablar con una persona"
→ Escalar inmediatamente. No intentar resolver primero.

Cliente: "Mi paquete llegó dañado y tengo fotos como prueba"
→ Verificar cliente → buscar orden → procesar reembolso. Es un caso estándar.

Cliente: "Quiero que igualen el precio que vi en otra tienda"
→ Escalar. La política solo cubre precios de nuestro propio sitio.
"""

## Paso 5 — Agentic loop completo

**Concepto clave:** El loop continúa mientras `stop_reason == "tool_use"`. Termina cuando `stop_reason == "end_turn"`.

In [ ]:
def run_support_agent(user_message: str) -> str:
    state = SupportAgentState()
    messages = [{"role": "user", "content": user_message}]
    
    print(f"\n👤 Cliente: {user_message}")
    print("-" * 50)
    
    iteration = 0
    while True:
        iteration += 1
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            tools=TOOLS,
            messages=messages
        )
        
        print(f"\n[Iteración {iteration}] stop_reason: {response.stop_reason}")
        
        if response.stop_reason == "end_turn":
            final_text = next(
                (b.text for b in response.content if hasattr(b, "text")), 
                "(sin respuesta de texto)"
            )
            print(f"\n🤖 Agente: {final_text}")
            return final_text
        
        elif response.stop_reason == "tool_use":
            # Agregar respuesta del asistente al historial
            messages.append({"role": "assistant", "content": response.content})
            
            # Ejecutar todas las tools solicitadas
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  → Tool: {block.name}({json.dumps(block.input, ensure_ascii=False)})")
                    result = state.execute_tool(block.name, block.input)
                    print(f"  ← Resultado: {result[:100]}..." if len(result) > 100 else f"  ← Resultado: {result}")
                    
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })
            
            # Agregar resultados al historial
            messages.append({"role": "user", "content": tool_results})
        
        else:
            print(f"⚠️ stop_reason inesperado: {response.stop_reason}")
            break

## Paso 6 — Probar los casos del examen

In [ ]:
# Caso 1: Reembolso estándar
result = run_support_agent(
    "Hola, soy maria@example.com. Mi paquete ORD-9872 llegó dañado. "
    "Quiero un reembolso por favor."
)

In [ ]:
# Caso 2: Escalación por pedido explícito
result = run_support_agent(
    "Soy maria@example.com. Quiero hablar con una persona real ahora."
)

In [ ]:
# Caso 3: Prueba del gate programático
# El agente debería verificar al cliente ANTES de buscar la orden
result = run_support_agent(
    "¿Cuál es el estado de mi orden ORD-9872?"
    # No provee identidad → el agente debe pedirla o el gate lo bloquea
)

## Preguntas de Reflexión

1. ¿Qué pasa si eliminás el gate programático y dejás solo la instrucción en el prompt? ¿Cuántas veces fallaría?
2. ¿Por qué el hook de intercepción para reembolsos > $500 es más confiable que una instrucción en el system prompt?
3. ¿Qué información incluiría el handoff estructurado si el caso escala a un humano?
4. ¿Cómo probarías que la descripción de `get_customer` vs `lookup_order` previene el mal enrutamiento?